# Day 6: Data Quality, Validation & Quarantine DLQ Engine

This notebook demonstrates running validation on intentional bad records verifying routing to quarantine DLQ.

In [ ]:
# Install pydeequ for data quality checks
!pip install pydeequ pyspark

In [ ]:
import os
os.environ['SPARK_VERSION'] = '3.3'
from pyspark.sql import SparkSession
import pydeequ

spark = SparkSession.builder \
    .appName("Day06_DQ_Circuit_Breaker") \
    .config("spark.jars.packages", pydeequ.deequ_maven_coord) \
    .config("spark.jars.excludes", pydeequ.f2j_maven_coord) \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark session with PyDeequ created.")

In [ ]:
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite

# Intentional bad records (Null person_id, negative concept_id)
data = [
    (1, "P001", 32020), # Good
    (2, None, 32020),   # Bad: Null person_id
    (3, "P002", -1)     # Bad: Negative concept
]
df = spark.createDataFrame(data, ["condition_occurrence_id", "person_id", "condition_concept_id"])

check = Check(spark, CheckLevel.Error, "OMOP Data Quality Check")

checkResult = VerificationSuite(spark) \
    .onData(df) \
    .addCheck(
        check.isComplete("person_id") \
        .isNonNegative("condition_concept_id")
    ) \
    .run()

checkResult_df = VerificationSuite.checkResultsAsDataFrame(spark, checkResult)
checkResult_df.show(truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Simulated Row-level DLQ Routing based on constraint failures
# In PyDeequ, row-level validation is typically done via column-level metrics or UDFs if complex.
# For demonstration, we simply filter the dataframe based on the rules we know failed.

good_records = df.filter(col("person_id").isNotNull() & (col("condition_concept_id") >= 0))
bad_records = df.filter(col("person_id").isNull() | (col("condition_concept_id") < 0))

print("Records routing to Silver:")
good_records.show()

print("Records routing to DLQ:")
bad_records.show()